In [ ]:
    ############    #############   Configuration management   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Configuration Management   #############   ##############   

 =>  12-factor apps keep config (DB URLs, API keys, feature flags) in the environment, never
       hard-coded, so the same image/code runs unmodified in dev, staging, and prod (see the
       Engineering Standards topic for the full 12-factor list).

 =>  pydantic-settings' BaseSettings reads and validates environment variables the same way
       a request body is validated -- missing/invalid config fails fast at startup, not at
       3am in production.


In [ ]:
import os
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="APP_")

    environment: str = "development"
    database_url: str = "postgresql://localhost/dev"
    max_retries: int = 3

os.environ["APP_ENVIRONMENT"] = "production"
os.environ["APP_MAX_RETRIES"] = "5"

settings = Settings()
print(settings)


In [ ]:
 =>  Notice APP_MAX_RETRIES="5" (a string, from the OS environment) is coerced to the
       int 5 by Pydantic's validation -- the rest of the app can trust settings.max_retries
       is always an int.

 =>  In pydantic v2, settings configuration goes through 'model_config = SettingsConfigDict(...)'
       -- the older 'class Config:' inner-class style is pydantic v1 syntax and is deprecated.

 =>  Never load secrets (API keys, DB passwords) via a plain .env committed to git -- use a
       secrets manager (Azure Key Vault / AWS Secrets Manager, covered in Phase 7).


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Add an .env file (git-ignored!) and set 'model_config = SettingsConfigDict(env_file=".env")'
           so Settings() reads it automatically.

 =>  [ ] Add a field_validator to Settings that rejects an invalid 'environment' value
           (only 'development', 'staging', 'production' allowed).


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Committing a real .env file with live secrets to git -- always .gitignore it and commit
       only a .env.example with placeholder values.

 =>  Reading os.environ directly scattered across the codebase instead of through one
       validated Settings object -- makes it impossible to know all your config surface at
       a glance, and skips validation entirely.
